In [1]:
%pip install peft


In [2]:
%pip install torch torchvision --index-url https://download.pytorch.org/whl/cu130

Looking in indexes: https://download.pytorch.org/whl/cu130


In [5]:
%pip install --upgrade torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 36.7 MB/s eta 0:00:00a 0:00:01
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
# 2. Create the symlink 
!ln -s "/content/drive/MyDrive/syntheticdata/checkpoints" "/content/checkpoints"


ln: failed to create symbolic link '/content/datasets/healthy2im': No such file or directory


In [11]:
!ln -s "/content/drive/MyDrive/syntheticdata/healthy2im" "/content/datasets/healthy2im"

ln: failed to create symbolic link '/content/datasets/healthy2im': No such file or directory


In [3]:
from diffusers import StableDiffusionInpaintPipeline, DDPMScheduler
import torch

model_id = "stable-diffusion-v1-5/stable-diffusion-inpainting"  # or stabilityai/stable-diffusion-2-inpainting

pipe = StableDiffusionInpaintPipeline.from_pretrained(model_id, torch_dtype=torch.float16)
unet = pipe.unet
vae = pipe.vae
text_encoder = pipe.text_encoder
tokenizer = pipe.tokenizer
noise_scheduler = DDPMScheduler.from_pretrained(model_id, subfolder="scheduler")

vae.requires_grad_(False)
text_encoder.requires_grad_(False)  # freeze text encoder too — LoRA the UNet only, standard practice
vae.eval()
text_encoder.eval()

device = "cuda"
vae.to(device, dtype=torch.float16)
text_encoder.to(device, dtype=torch.float16)
unet.to(device, dtype=torch.float32)  # train UNet in fp32 for stability, or use fp16 + gradient scaling

Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


model_index.json:   0%|          | 0.00/548 [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 16 files:   0%|          | 0/16 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/396 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

An error occurred while trying to fetch /root/.cache/huggingface/hub/models--stable-diffusion-v1-5--stable-diffusion-inpainting/snapshots/8a4288a76071f7280aedbdb3253bdb9e9d5d84bb/unet: Error no file named diffusion_pytorch_model.safetensors found in directory /root/.cache/huggingface/hub/models--stable-diffusion-v1-5--stable-diffusion-inpainting/snapshots/8a4288a76071f7280aedbdb3253bdb9e9d5d84bb/unet.
Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.
An error occurred while trying to fetch /root/.cache/huggingface/hub/models--stable-diffusion-v1-5--stable-diffusion-inpainting/snapshots/8a4288a76071f7280aedbdb3253bdb9e9d5d84bb/vae: Error no file named diffusion_pytorch_model.safetensors found in directory /root/.cache/huggingface/hub/models--stable-diffusion-v1-5--stable-diffusion-inpainting/snapshots/8a4288a76071f7280aedbdb3253bdb9e9d5d84bb/vae.
Defaulting to unsafe serialization. Pass `allow_pickle=False` to raise an error instead.
/usr/local/lib

UNet2DConditionModel(
  (conv_in): Conv2d(9, 320, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (time_proj): Timesteps()
  (time_embedding): TimestepEmbedding(
    (linear_1): Linear(in_features=320, out_features=1280, bias=True)
    (act): SiLU()
    (linear_2): Linear(in_features=1280, out_features=1280, bias=True)
  )
  (down_blocks): ModuleList(
    (0): CrossAttnDownBlock2D(
      (attentions): ModuleList(
        (0-1): 2 x Transformer2DModel(
          (norm): GroupNorm(32, 320, eps=1e-06, affine=True)
          (proj_in): Conv2d(320, 320, kernel_size=(1, 1), stride=(1, 1))
          (transformer_blocks): ModuleList(
            (0): BasicTransformerBlock(
              (norm1): LayerNorm((320,), eps=1e-05, elementwise_affine=True)
              (attn1): Attention(
                (to_q): Linear(in_features=320, out_features=320, bias=False)
                (to_k): Linear(in_features=320, out_features=320, bias=False)
                (to_v): Linear(in_features=320, out_fe

In [6]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,
    lora_alpha=16,
    target_modules=["to_q", "to_k", "to_v", "to_out.0"],
    lora_dropout=0.05,
)

unet = get_peft_model(unet, lora_config)
unet.print_trainable_parameters()

trainable params: 3,188,736 || all params: 862,724,100 || trainable%: 0.3696


In [12]:
from torch.utils.data import Dataset
from torchvision import transforms
from PIL import Image
import numpy as np
import os, json
from pathlib import Path

class GastricIMInpaintDataset(Dataset):
    def __init__(self, image_dir, mask_dir, tokenizer, size=224):
        # captions_json maps filename -> caption string, e.g.
        # "img001.png": "intestinal metaplasia, gastric antrum, incomplete type"
        # with open(captions_json) as f:
        #     self.captions = json.load(f)
        self.image_dir = image_dir
        directory = Path(image_dir)
        self.filenames = [f.name for f in directory.iterdir() if f.is_file()]
        self.mask_dir = mask_dir
        # self.filenames = list(self.captions.keys())
        self.captions = {fname: "intestinal metaplasia, gastric mucosa" for fname in self.filenames}
        self.tokenizer = tokenizer
        self.image_transform = transforms.Compose([
            transforms.Resize((size, size)),
            transforms.ToTensor(),
            transforms.Normalize([0.5]*3, [0.5]*3),
        ])
        self.mask_transform = transforms.Compose([
            transforms.Resize((size, size)),
            transforms.ToTensor(),
        ])

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, idx):
        fname = self.filenames[idx]
        image = Image.open(os.path.join(self.image_dir, fname)).convert("RGB")
        mask = Image.open(os.path.join(self.mask_dir, fname)).convert("L")  # gland mask, white = region to inpaint

        image_t = self.image_transform(image)
        mask_t = self.mask_transform(mask)
        mask_t = (mask_t > 0.5).float()

        masked_image_t = image_t * (1 - mask_t)  # black out masked region

        caption = self.captions[fname]
        input_ids = self.tokenizer(
            caption, padding="max_length", truncation=True,
            max_length=self.tokenizer.model_max_length, return_tensors="pt"
        ).input_ids[0]

        return {
            "pixel_values": image_t,
            "mask": mask_t,
            "masked_image": masked_image_t,
            "input_ids": input_ids,
        }

# dataset = GastricIMInpaintDataset(
#     "../datasets/healthy2im/trainB",
#     "../datasets/healthy2im/masks",
#     tokenizer,
# )

dataset = GastricIMInpaintDataset(
    "./healthy2im/trainB",
    "./healthy2im/masks",
    tokenizer,
)

In [13]:
from torch.utils.data import DataLoader
dataloader = DataLoader(dataset, batch_size=4, shuffle=True, num_workers=4)

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


In [14]:
optimizer = torch.optim.AdamW(
    [p for p in unet.parameters() if p.requires_grad],
    lr=1e-4, weight_decay=1e-6
)

from diffusers.optimization import get_cosine_schedule_with_warmup
num_epochs = 15
lr_scheduler = get_cosine_schedule_with_warmup(
    optimizer, num_warmup_steps=100,
    num_training_steps=len(dataloader) * num_epochs,
)

scaling_factor = vae.config.scaling_factor

for epoch in range(num_epochs):
    print("Starting epoch", epoch)
    for step, batch in enumerate(dataloader):
        images = batch["pixel_values"].to(device, dtype=torch.float16)
        masks = batch["mask"].to(device, dtype=torch.float16)
        masked_images = batch["masked_image"].to(device, dtype=torch.float16)
        input_ids = batch["input_ids"].to(device)

        with torch.no_grad():
            latents = vae.encode(images).latent_dist.sample() * scaling_factor
            masked_latents = vae.encode(masked_images).latent_dist.sample() * scaling_factor
            mask_latent = torch.nn.functional.interpolate(masks, size=latents.shape[-2:])
            encoder_hidden_states = text_encoder(input_ids)[0]

        noise = torch.randn_like(latents)
        bsz = latents.shape[0]
        timesteps = torch.randint(0, noise_scheduler.config.num_train_timesteps, (bsz,), device=device).long()
        noisy_latents = noise_scheduler.add_noise(latents, noise, timesteps)

        # SD Inpainting's UNet expects 9 channels: noisy_latents(4) + mask(1) + masked_latents(4)
        unet_input = torch.cat([noisy_latents, mask_latent, masked_latents], dim=1).to(torch.float32)

        noise_pred = unet(
            unet_input, timesteps,
            encoder_hidden_states=encoder_hidden_states.to(torch.float32)
        ).sample

        # supervise loss primarily inside the masked region
        loss = torch.nn.functional.mse_loss(
            noise_pred.float() * mask_latent, noise.float() * mask_latent
        )
        # optionally add a small full-image loss term to keep boundary coherence:
        # loss += 0.1 * F.mse_loss(noise_pred.float(), noise.float())

        loss.backward()
        torch.nn.utils.clip_grad_norm_(unet.parameters(), 1.0)
        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()

        if step % 50 == 0:
            print(f"epoch {epoch} step {step} loss {loss.item():.4f}")

    unet.save_pretrained(f"../checkpoints/lora_gastric_im_inpaint/epoch_{epoch}/")

Starting epoch 0


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


FileNotFoundError: Caught FileNotFoundError in DataLoader worker process 0.
Original Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/worker.py", line 358, in _worker_loop
    data = fetcher.fetch(index)  # type: ignore[possibly-undefined]
           ^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/utils/data/_utils/fetch.py", line 54, in fetch
    data = [self.dataset[idx] for idx in possibly_batched_index]
            ~~~~~~~~~~~~^^^^^
  File "/tmp/ipykernel_1472/3747525098.py", line 37, in __getitem__
    mask = Image.open(os.path.join(self.mask_dir, fname)).convert("L")  # gland mask, white = region to inpaint
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/PIL/Image.py", line 3513, in open
    fp = builtins.open(filename, "rb")
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: './healthy2im/masks/

In [ ]:
from peft import get_peft_model_state_dict
lora_state_dict = get_peft_model_state_dict(unet)
torch.save(lora_state_dict, "../checkpoints/lora_gastric_im_inpaint_final.pt")

In [ ]:
from diffusers import StableDiffusionInpaintPipeline
from PIL import Image

pipe = StableDiffusionInpaintPipeline.from_pretrained(model_id, torch_dtype=torch.float16).to("cuda")
pipe.unet = unet  # your LoRA-adapted unet (merge_and_unload() first if you want a plain UNet object)

image = Image.open("healthy_gastric_patch.png").convert("RGB")
mask = Image.open("healthy_gland_mask.png").convert("L")  # white = region to convert to IM

result = pipe(
    prompt="intestinal metaplasia, gastric antrum, goblet cells, incomplete type",
    image=image,
    mask_image=mask,
    strength=0.99,        # near-1.0 = fully regenerate masked region from noise
    guidance_scale=7.5,
    num_inference_steps=50,
).images[0]

result.save("edited_gastric_patch.png")